In [ ]:
import evaluate
import numpy as np
import pandas as pd
from datasets import Dataset
from sklearn.metrics import confusion_matrix
from transformers import AutoModelForSequenceClassification,AutoTokenizer,DataCollatorWithPadding,Trainer,TrainingArguments



In [ ]:
df = pd.read_csv("spam.csv", encoding='latin1')

In [ ]:
checkpoint = "bert-base-uncased"
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

In [ ]:
df_tl = df.copy()
df_tl.columns = ["label", "text"]
df_tl["label"] = df_tl["label"].map({"ham": 0, 
                                       "spam": 1})
ds = Dataset.from_pandas(df_tl)

ds = ds.train_test_split(test_size=0.2)
train_dataset = ds["train"]
test_dataset = ds["test"]

In [ ]:
def tokenize_function(example):
    return tokenizer(example["text"], truncation=True, padding=True)


tokenizer = AutoTokenizer.from_pretrained(checkpoint)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [ ]:
tokenized_train, tokenized_test

In [ ]:
def compute_metrics(eval_preds):
    accuracy_metric = evaluate.load("accuracy")
    f1_metric = evaluate.load("f1")
    precision_metric = evaluate.load("precision")
    recall_metric = evaluate.load("recall")

    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    cm = confusion_matrix(labels, predictions)

    return {
        "accuracy": accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"],
        "f1": f1_metric.compute(predictions=predictions, references=labels, average="binary")["f1"],
        "precision": precision_metric.compute(predictions=predictions, references=labels, average="binary")["precision"],
        "recall": recall_metric.compute(predictions=predictions, references=labels, average="binary")["recall"],
        "confusion_matrix": cm.tolist()
    }

In [ ]:
training_args = TrainingArguments("test", eval_strategy="epoch")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

trainer = Trainer(
    model,
    training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
trainer.train()